# Renewed Fooocus — Verified One-Cell Google Colab

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Optionally paste exact Civitai model-version URLs or numeric IDs.
3. Press the play button once.

The launcher creates a managed Python 3.10 environment, verifies PyTorch can access the GPU, optionally mounts Google Drive, installs requested Civitai assets, and launches one Renewed Fooocus interface.


In [ ]:
#@title ▶ Run Renewed Fooocus
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
PRESET = "realistic" #@param ["realistic", "anime", "default"]
CIVITAI_CHECKPOINTS = "" #@param {type:"string"}
CIVITAI_LORAS = "" #@param {type:"string"}
CIVITAI_API_TOKEN = "" #@param {type:"string"}

# Civitai examples:
# Checkpoint URL: https://civitai.com/models/123/name?modelVersionId=456
# Multiple entries: 456, 789, https://civitai.com/api/download/models/101112

import os
import pathlib
import shutil
import subprocess
import sys
import time

REPO = "https://github.com/JaeTheOP/fooocus-clone.git"
BRANCH = "agent/civitai-model-manager"
ROOT = pathlib.Path("/content/renewed-fooocus")

last_error = None
for attempt in range(1, 4):
    if ROOT.exists():
        shutil.rmtree(ROOT)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(ROOT)],
            check=True,
        )
        last_error = None
        break
    except subprocess.CalledProcessError as exc:
        last_error = exc
        print(f"Git clone attempt {attempt}/3 failed.")
        if attempt < 3:
            time.sleep(3)
if last_error is not None:
    raise RuntimeError("Could not clone Renewed Fooocus after three attempts.") from last_error

commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Renewed Fooocus revision:", commit)

env = os.environ.copy()
env["RF_PRESET"] = "" if PRESET == "default" else PRESET
env["RF_CIVITAI_CHECKPOINTS"] = CIVITAI_CHECKPOINTS.strip()
env["RF_CIVITAI_LORAS"] = CIVITAI_LORAS.strip()
env["RF_CIVITAI_TOKEN"] = CIVITAI_API_TOKEN.strip()
env["PYTHONUNBUFFERED"] = "1"
env["UV_LINK_MODE"] = "copy"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    drive_root = pathlib.Path("/content/drive/MyDrive/Renewed Fooocus")
    drive_root.mkdir(parents=True, exist_ok=True)
    env["RF_DRIVE_ROOT"] = str(drive_root)
else:
    env.pop("RF_DRIVE_ROOT", None)

result = subprocess.run(
    [sys.executable, "-u", "colab_launcher.py"],
    cwd=ROOT,
    env=env,
)
if result.returncode not in (0, 130):
    raise RuntimeError(
        "Renewed Fooocus startup failed. Read the detailed error printed immediately above this line."
    )


## What the startup check verifies

Before Fooocus launches, the notebook verifies the NVIDIA GPU, Python 3.10, PyTorch, Torchvision, CUDA access, Gradio, OpenCV, HTTPX, and NumPy. A failed check stops startup and prints the exact error instead of opening a partially working interface.

Use the exact Civitai model-version URL or numeric version ID. Put checkpoints in **CIVITAI_CHECKPOINTS** and LoRAs in **CIVITAI_LORAS**. Separate multiple entries with commas. Requested downloads are treated as required: an invalid or failed model download stops startup with an explanation.

Leave Google Drive disabled for the fastest first test. Enable it after the first successful launch to persist models and outputs under `MyDrive/Renewed Fooocus`.

When startup finishes, open the `gradio.live` URL printed near the bottom of the cell output. Keep the cell running while using Renewed Fooocus.
